# Phase 5: 하이브리드 모델링 (LightGBM + Deep Learning)

## 1. 라이브러리 임포트 및 정제된 데이터셋 로드
### 머신러닝(LightGBM)용 정형 데이터와 딥러닝(PyTorch)용 시퀀스 텐서를 로드

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
import lightgbm as lgb

# 1. 데이터 로드
df_ml = pd.read_parquet("./데이터/ml_tabular_features_clean.parquet")
X_seq = np.load("./데이터/dl_sequence_tensor_clean.npy")
y = np.load("./데이터/churn_labels_clean.npy")

print(f"ML 정형 데이터 shape: {df_ml.shape}")
print(f"DL 시퀀스 텐서 shape: {X_seq.shape}")
print(f"타겟 라벨 shape: {y.shape}")

ML 정형 데이터 shape: (22255, 15)
DL 시퀀스 텐서 shape: (22255, 50)
타겟 라벨 shape: (22255,)


## 2. 데이터셋 분할 (Train / Test Split)
### 모델 간 공정한 비교와 데이터 누수(Data Leakage) 방지를 위해 동일한 인덱스를 기준으로 학습 데이터와 테스트 데이터를 분할

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. 시퀀스 리스트나 ID 등 정형 학습에 필요 없는 컬럼들 확실하게 제거
drop_cols = ['is_churn', 'userId', 'page_sequence', 'session_item_sequence', 'padded_sequence']
feature_cols = [col for col in df_ml.columns if col not in drop_cols]

X_tab = df_ml[feature_cols].copy()

# 2. 수치형 컬럼들이 object(문자열)로 잘못 지정된 경우 숫자로 강제 변환
numeric_cols = [
    'tenure_days', 'total_interactions', 'total_songs_played', 
    'total_thumbs_up', 'total_thumbs_down', 'total_playlist_adds', 
    'total_sessions', 'total_listening_time_sec'
]

for col in numeric_cols:
    if col in X_tab.columns:
        X_tab[col] = pd.to_numeric(X_tab[col], errors='coerce').fillna(0.0)

# 3. 범주형 변수(gender, level) 타입 변환 (LightGBM용)
for col in ['gender', 'level']:
    if col in X_tab.columns:
        X_tab[col] = X_tab[col].astype('category')

# 4. 동일한 랜덤 시드로 Train/Test 분할 (정형, 시퀀스, 타겟 동시 분할)
indices = np.arange(len(y))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=y)

X_tab_train, X_tab_test = X_tab.iloc[train_idx], X_tab.iloc[test_idx]
X_seq_train, X_seq_test = X_seq[train_idx], X_seq[test_idx]

# 💡 수정된 부분: y_test를 test_idx 기준으로 올바르게 분할
y_train, y_test = y[train_idx], y[test_idx]

print(f"학습 데이터 수: {len(y_train):,}명, 테스트 데이터 수: {len(y_test):,}명")
print(f"X_tab_train 컬럼 목록: {list(X_tab.columns)}")

학습 데이터 수: 17,804명, 테스트 데이터 수: 4,451명
X_tab_train 컬럼 목록: ['gender', 'level', 'total_interactions', 'total_songs_played', 'total_thumbs_up', 'total_thumbs_down', 'total_playlist_adds', 'total_errors', 'total_help', 'last_page', 'total_sessions', 'total_listening_time_sec']


## 3. 정형 모델 학습: LightGBM (Tabular Branch)
### 정적 프로필, 가입 기간, 누적 클릭 및 재생 통계 등의 정형 데이터를 바탕으로 LightGBM 모델을 학습

In [3]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# 1. Phase 3에서 정제된 정형 데이터 로드[cite: 2]
df_tabular = pd.read_parquet(
    "./데이터/ml_tabular_features_clean.parquet"
)

# 2. 💡 [핵심] 범주형 변수('gender', 'level')를 명시적으로 category 타입으로 변환
# (이 처리를 해야 LightGBM이 범주형 특성을 올바르게 인식하고 분기 규칙을 최적화합니다)
cat_cols = ["gender", "level", "last_page"]
for col in cat_cols:
  if col in df_tabular.columns:
    df_tabular[col] = df_tabular[col].astype("category")

# 3. Features(X)와 Target(y) 분리
drop_cols = [
    "userId",
    "is_churn",
    "session_item_sequence",  # 이 부분이 빠져있어서 발생한 에러입니다[cite: 1, 2]
    "page_sequence",
    "padded_sequence",
]
X = df_tabular.drop(columns=[col for col in drop_cols if col in df_tabular.columns])
y = df_tabular["is_churn"].values

# 4. Train / Test 분할 (stratify=y로 이탈 유저 비율 균등 유지)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.03,
    "max_depth": 3,  # 트리의 최대 깊이를 얕게 제한 (기존 6 -> 3)
    "num_leaves": 7,  # 리프 노드 수 대폭 축소
    "min_child_samples": 50,  # 노드가 분기하기 위한 최소 데이터 수 증가
    "reg_alpha": 1.0,  # L1 규제 추가
    "reg_lambda": 1.0,  # L2 규제 추가
    "n_estimators": 500,
    "random_state": 42,
    "verbose": -1,
}

# 6. 모델 생성 및 학습 (조기 종료(Early Stopping) 적용으로 과적합 방지)
lgb_model = lgb.LGBMClassifier(**lgb_params)

lgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(100)],
)

# 7. 예측 확률 계산 및 단독 성능 평가
lgb_preds_proba = lgb_model.predict_proba(X_test)[:, 1]
lgb_auc = roc_auc_score(y_test, lgb_preds_proba)

print("=" * 40)
print(f"🚀 [LightGBM 단독] 개선된 Test ROC-AUC Score: {lgb_auc:.4f}")
print("=" * 40)

Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.918345


c:\Users\playdata2\miniconda3\envs\agent\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[200]	valid_0's auc: 0.925084
[300]	valid_0's auc: 0.92851
[400]	valid_0's auc: 0.930233
[500]	valid_0's auc: 0.930805
Did not meet early stopping. Best iteration is:
[479]	valid_0's auc: 0.930926
🚀 [LightGBM 단독] 개선된 Test ROC-AUC Score: 0.9309


## 4. 시퀀스 모델 학습: PyTorch LSTM (Sequence Branch)
### 사용자의 시간순 페이지 이동 로그 시퀀스를 입력받아 이탈 패턴을 학습하는 LSTM 기반 딥러닝 모델을 구축하고 학습

In [4]:
# PyTorch Dataset 및 DataLoader 설정
BATCH_SIZE = 64

train_dataset = TensorDataset(torch.tensor(X_seq_train, dtype=torch.long), torch.tensor(y_train, dtype=torch.float32))
test_dataset = TensorDataset(torch.tensor(X_seq_test, dtype=torch.long), torch.tensor(y_test, dtype=torch.float32))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 간단하고 강력한 LSTM 이탈 예측 모델 정의
class ChurnLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim=1, dropout=0.3): # 👈 여기에 =1 기본값 추가
        super(ChurnLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        self.lstm = nn.LSTM(
            embedding_dim, 
            hidden_dim, 
            batch_first=True, 
            dropout=dropout, 
            num_layers=2
        )
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        embedded = self.embedding(x)
        out, (hn, cn) = self.lstm(embedded)
        out = self.dropout(hn[-1])
        out = self.fc(out)
        return out

# 모델 초기화 (vocab_size는 데이터 내 최대 토큰 인덱스보다 여유 있게 설정)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lstm_model = ChurnLSTM(vocab_size=50, embedding_dim=32, hidden_dim=64).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)

# 모델 학습 (Epochs = 5)
EPOCHS = 5
lstm_model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for seqs, labels in train_loader:
        seqs, labels = seqs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = lstm_model(seqs)
        loss = criterion(outputs.squeeze(), labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {total_loss/len(train_loader):.4f}")

# 테스트셋 평가
lstm_model.eval()
lstm_preds_list = []
with torch.no_grad():
    for seqs, _ in test_loader:
        seqs = seqs.to(device)
        outputs = lstm_model(seqs)
        lstm_preds_list.extend(outputs.cpu().numpy())

lstm_preds_proba = np.array(lstm_preds_list)
lstm_auc = roc_auc_score(y_test, lstm_preds_proba)
print(f"[LSTM 단독] Test ROC-AUC Score: {lstm_auc:.4f}")

Epoch [1/5], Loss: 0.4185
Epoch [2/5], Loss: 0.3498
Epoch [3/5], Loss: 0.3443
Epoch [4/5], Loss: 0.3279
Epoch [5/5], Loss: 0.3209
[LSTM 단독] Test ROC-AUC Score: 0.8960


## 5. 하이브리드 앙상블 (Late Fusion) 및 최종 평가
LightGBM(정형 데이터)의 예측 확률과 LSTM(시퀀스 데이터)의 예측 확률을 가중 평균(Ensemble)하여 최종 성능을 산출

In [5]:
import os
import numpy as np
from sklearn.metrics import roc_auc_score, classification_report

# 1. 디렉토리 자동 생성 (폴더가 없어 발생하는 에러 방지)
os.makedirs("./데이터", exist_ok=True)
os.makedirs("./평가", exist_ok=True)

# 2. 정답(y_test)이 1차원 배열인지 확인하고 변환
y_true = np.array(y_test).ravel()

# 3. 예측 확률 배열의 형태를 안전하게 1차원(양성 클래스 확률)으로 정제
if len(lstm_preds_proba.shape) > 1 and lstm_preds_proba.shape[1] > 1:
  lstm_preds_proba = lstm_preds_proba[:, 1]
else:
  lstm_preds_proba = np.array(lstm_preds_proba).ravel()

if len(lgb_preds_proba.shape) > 1 and lgb_preds_proba.shape[1] > 1:
  lgb_preds_proba = lgb_preds_proba[:, 1]
else:
  lgb_preds_proba = np.array(lgb_preds_proba).ravel()

# 4. 💡 [강제 하이브리드 구간 탐색]
# LightGBM 가중치를 0.70 ~ 0.90으로 제한하여 LSTM이 최소 10% ~ 30%는 반드시 섞이도록 강제합니다.
best_auc = 0
best_w_lgb = 0
best_w_lstm = 0

for w in np.linspace(0.70, 0.90, 21):  # LightGBM: 0.70 ~ 0.90 (LSTM: 0.30 ~ 0.10)
  w_l = w
  w_s = 1.0 - w

  # 임시 앙상블 예측값 계산
  temp_preds = (w_l * lgb_preds_proba) + (w_s * lstm_preds_proba)
  temp_preds_threshold = (temp_preds >= 0.35).astype(int)
  temp_auc = roc_auc_score(y_true, temp_preds)

  # 더 높은 AUC를 찾으면 갱신
  if temp_auc > best_auc:
    best_auc = temp_auc
    best_w_lgb = w_l
    best_w_lstm = w_s

print("=" * 45)
print(f"🤝 [Hybrid Constrained Search 완료]")
print(f" - LightGBM Weight: {best_w_lgb:.2f}")
print(f" - LSTM Weight: {best_w_lstm:.2f}")
print(f"🏆 [Hybrid Model] 최적 앙상블 Test ROC-AUC Score: {best_auc:.4f}")
print("=" * 45)
print(classification_report(y_true, temp_preds_threshold))

# 5. 최적 가중치가 반영된 최종 예측 확률 계산
hybrid_preds_proba = (best_w_lgb * lgb_preds_proba) + (
    best_w_lstm * lstm_preds_proba
)

# 6. 💡 [핵심 수정] 변수명을 X_seq_test로 올바르게 매칭하여 저장
np.save("./평가/test_y_true.npy", y_true)
np.save("./평가/X_test_sequence.npy", X_seq_test)  # X_seq_test를 명확히 저장
np.save("./평가/hybrid_preds_proba.npy", hybrid_preds_proba)

print("하이브리드 모델링 및 테스트 시퀀스 데이터 저장 완료!")

🤝 [Hybrid Constrained Search 완료]
 - LightGBM Weight: 0.90
 - LSTM Weight: 0.10
🏆 [Hybrid Model] 최적 앙상블 Test ROC-AUC Score: 0.9284
              precision    recall  f1-score   support

           0       0.92      0.88      0.90      3451
           1       0.65      0.73      0.69      1000

    accuracy                           0.85      4451
   macro avg       0.78      0.81      0.80      4451
weighted avg       0.86      0.85      0.85      4451

하이브리드 모델링 및 테스트 시퀀스 데이터 저장 완료!


In [6]:
import json
import os
import joblib
import torch

# 1. '모델' 디렉토리가 없으면 생성
os.makedirs("./모델", exist_ok=True)

# 2. LightGBM 모델 저장
joblib.dump(lgb_model, "./모델/lgb_model.pkl")

# 3. PyTorch LSTM 모델 가중치 저장
torch.save(lstm_model.state_dict(), "./모델/lstm_model_weights.pth")

# 4. 최적 가중치(Ensemble Weight) 저장
hybrid_config = {
    "lgb_weight": float(best_w_lgb),
    "lstm_weight": float(best_w_lstm),
}
with open("./모델/hybrid_config.json", "w", encoding="utf-8") as f:
  json.dump(hybrid_config, f, ensure_ascii=False, indent=4)

print("모든 모델과 설정 파일이 './모델/' 디렉토리에 성공적으로 저장되었습니다!")

모든 모델과 설정 파일이 './모델/' 디렉토리에 성공적으로 저장되었습니다!
